In [ ]:
from src.preprocess.parquet_preprocessor import ParquetPreprocessor

ParquetPreprocessor.csv_to_parquet("data/raw/active_alarms_prod.csv", "data/raw/active_alarms_prod.parquet")

In [1]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")

from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/active_alarms_prod.parquet")
graph_repo = AlarmGraphRepository(os.getenv("ACTIVE_DB_PATH"))

In [2]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationActive

SimpleTimeCorrelationActive.train(lazy_frame, graph_repo)

Processando nós: 100%|██████████| 866/866 [00:13<00:00, 63.69nó/s] 


In [4]:
from src.postprocess.enumerate_incidents import EnumerateIncidents

EnumerateIncidents.enumerate_data(graph_repo)

graph_repo.preview_nodes()

Calculando WCC: 100%|██████████| 866/866 [00:03<00:00, 288.51nó/s]


alert_id,node_id,alert_type,start_time,end_time,incident
str,str,str,date,date,i32
"""305da354-2edf-444e-a2b7-4edb06…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_QLE…",2025-12-05,2025-12-05,0
"""8a6acea4-c8db-46f8-8c6a-0ca91a…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_CUR""",2025-12-05,2025-12-05,0
"""514ed2e4-e107-4d2b-82ba-a232c6…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_AVG""",2025-12-05,2025-12-05,0
"""4ef9274c-e078-468f-8b3f-dc44d3…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_PHYSICAL_DISK_ALLOCATI…",2025-11-11,2025-12-05,0
"""88b82f4c-e0a0-4097-844a-84d5b3…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IOSZ_AVG""",2025-12-05,2025-12-05,0
…,…,…,…,…,…
"""a9518d63-8bd3-40eb-ab46-710461…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_KBYTES_P…",2025-12-05,2025-12-05,0
"""d046d918-6938-4705-9caf-672fbd…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_IO_PER_S…",2025-12-05,2025-12-05,0
"""08d790f7-1153-456b-983f-80d267…","""11dabbf8-b36c-42e6-9695-51ac6c…","""STORAGE_VOLUME_STATUS_KBYTES_P…",2025-12-05,2025-12-05,0


In [ ]:
from src.utils.node_summary import node_summary

summary, general_metrics = node_summary(os.getenv("ACTIVE_DB_PATH"))

with pl.Config(tbl_rows=-1):
    display(summary)